# Is Germany still Europe's engine? Rebuilding a viral GDP chart from Eurostat

A chart circulating on social media shows real GDP for Germany, Spain, Italy and France indexed to 100 at a pre-pandemic base, with Spain at about 112 and Germany at about 102. That chart answers one question well (**how much has each economy grown since the base period?**) and hides another (**how big is each economy?**).

In this notebook we will:

1. Find Europe's 10 largest economies in Eurostat's data.
2. Pull quarterly GDP for them from the Eurostat API and rebuild the indexed chart.
3. Plot GDP **levels** and GDP **growth rates**.
4. Check the story on a **per person** basis.
5. Test a claim: *Spain, Türkiye and Poland together rival Germany in size and far outgrow it.*

**Key idea: chain-linked volumes.** Eurostat's "chain-linked volumes" (unit codes like `CLV20_MEUR`) are real GDP: output valued at a reference year's prices, so inflation is removed. Dividing every period by the base period and multiplying by 100 gives the index in the viral chart. Current prices (`CP_MEUR`) are nominal GDP, the better measure of an economy's size in euros today.

**Coverage note.** Eurostat covers EU members, EFTA countries (Switzerland, Norway, Iceland) and candidate countries such as Türkiye. It stopped publishing UK data after Brexit and never covered Russia, so "top 10" here means top 10 among the countries Eurostat reports. Section 7 shows how to add the UK from its own statistics office.

In [ ]:
import io
import urllib.request

import polars as pl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

TOP_N = 10            # how many of the largest economies to compare
PIN = None            # or a list like ["DE", "ES", "FR", "IT"] to skip the ranking
START = "2015-Q1"     # first quarter to download
BASE = "2019-Q4"      # quarter set to 100 (last full quarter before COVID)
HIGHLIGHT = "Germany" # drawn in bold on the charts

BLOC = ["ES", "TR", "PL"]  # the group we compare against Germany in section 6
BENCH = "DE"

NAMES = {
    "AT": "Austria", "BE": "Belgium", "BG": "Bulgaria", "CH": "Switzerland", "CY": "Cyprus",
    "CZ": "Czechia", "DE": "Germany", "DK": "Denmark", "EE": "Estonia", "EL": "Greece",
    "ES": "Spain", "FI": "Finland", "FR": "France", "HR": "Croatia", "HU": "Hungary",
    "IE": "Ireland", "IS": "Iceland", "IT": "Italy", "LT": "Lithuania", "LU": "Luxembourg",
    "LV": "Latvia", "MT": "Malta", "NL": "Netherlands", "NO": "Norway", "PL": "Poland",
    "PT": "Portugal", "RO": "Romania", "RS": "Serbia", "SE": "Sweden", "SI": "Slovenia",
    "SK": "Slovakia", "TR": "Türkiye", "UK": "United Kingdom", "ME": "Montenegro",
    "MK": "North Macedonia", "AL": "Albania", "BA": "Bosnia and Herzegovina", "XK": "Kosovo",
}

## 1. Fetching data from Eurostat

Eurostat exposes every dataset through an SDMX API:

```
.../sdmx/2.1/data/<dataset>/<key>?format=SDMX-CSV&startPeriod=2015-Q1
```

The **key** is a dot-separated filter, one slot per dimension. For quarterly GDP (`namq_10_gdp`) the dimensions are `freq.unit.s_adj.na_item.geo`; for annual GDP (`nama_10_gdp`) they are `freq.unit.na_item.geo`. A blank slot means "everything", and `+` joins several values (`DE+ES+FR`).

We leave `unit` blank on purpose. Eurostat periodically moves its reference year (2010, 2015, 2020...), so instead of hard-coding a unit code we download them all and pick the newest chain-linked one.

In [ ]:
EUROSTAT = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data"

def fetch_eurostat(dataset, key, start=None):
    """Download a Eurostat dataset slice as a Polars DataFrame."""
    url = f"{EUROSTAT}/{dataset}/{key}?format=SDMX-CSV"
    if start:
        url += f"&startPeriod={start}"
    print("GET", url)
    with urllib.request.urlopen(url, timeout=120) as resp:
        raw = resp.read()
    df = pl.read_csv(io.BytesIO(raw), infer_schema_length=0)  # read as text first
    df = df.rename({c: c.lower().replace(" ", "_") for c in df.columns})
    return df.with_columns(pl.col("obs_value").cast(pl.Float64, strict=False))

def newest_unit(units, prefix, suffix):
    """Pick e.g. CLV20_MEUR over CLV15_MEUR and CLV10_MEUR."""
    cands = [u for u in units if u.startswith(prefix) and u.endswith(suffix)
             and u[len(prefix):-len(suffix)].isdigit()]
    if not cands:
        raise ValueError(f"No {prefix}xx{suffix} unit in {sorted(units)}")
    return max(cands, key=lambda u: int(u[len(prefix):-len(suffix)]))

def name_of(col="geo"):
    return pl.col(col).replace(NAMES)

## 2. Who are the top 10?

We download **annual** GDP for every country Eurostat covers, rank by nominal GDP in euros, and keep the top `TOP_N`. Two-letter geo codes are countries; longer codes (`EU27_2020`, `EA20`) are aggregates, so we drop them. Each country is ranked on its most recent year, as long as that year is no more than one year behind Germany's (some countries report later than others).

In [ ]:
annual = (
    fetch_eurostat("nama_10_gdp", "A..B1GQ.", start="2014")
    .filter(pl.col("geo").str.len_chars() == 2)
    .drop_nulls("obs_value")
    .with_columns(year=pl.col("time_period").cast(pl.Int32))
)
A_REAL = newest_unit(annual["unit"].unique().to_list(), "CLV", "_MEUR")
latest_year = annual.filter((pl.col("geo") == "DE") & (pl.col("unit") == "CP_MEUR"))["year"].max()

ranking = (
    annual.filter((pl.col("unit") == "CP_MEUR") & (pl.col("year") >= latest_year - 1))
    .sort("year")
    .group_by("geo").agg(pl.col("year").last(), nominal_meur=pl.col("obs_value").last())
    .sort("nominal_meur", descending=True)
    .with_columns(
        rank=pl.int_range(1, pl.len() + 1),
        country=name_of(),
        nominal_eur_tn=(pl.col("nominal_meur") / 1e6).round(2),
    )
)
COUNTRIES = {g: NAMES.get(g, g) for g in (PIN or ranking["geo"].head(TOP_N).to_list())}
print("Comparing:", ", ".join(COUNTRIES.values()))
ranking.select("rank", "country", "year", "nominal_eur_tn").head(15)

## 3. Quarterly GDP for the top 10

In [ ]:
geo_key = "+".join(COUNTRIES)
raw = fetch_eurostat("namq_10_gdp", f"Q..SCA.B1GQ.{geo_key}", start=START)

units = raw["unit"].unique().to_list()
REAL_UNIT = newest_unit(units, "CLV", "_MEUR")   # real GDP, million euro
NOMINAL_UNIT = "CP_MEUR"                         # nominal GDP, million euro
print("Using real unit:", REAL_UNIT)

have = set(raw.filter(pl.col("unit") == REAL_UNIT).drop_nulls("obs_value")["geo"].unique())
missing = [NAMES.get(g, g) for g in COUNTRIES if g not in have]
if missing:
    print("No quarterly seasonally adjusted real GDP for:", ", ".join(missing),
          "(they still appear in the annual analysis in section 6)")

In [ ]:
quarter_to_date = pl.date(
    pl.col("time_period").str.slice(0, 4).cast(pl.Int32),
    pl.col("time_period").str.slice(-1).cast(pl.Int32) * 3 - 2,
    1,
)

gdp = (
    raw
    .filter(pl.col("unit").is_in([REAL_UNIT, NOMINAL_UNIT]))
    .with_columns(country=name_of(), date=quarter_to_date)
    .select("country", "geo", "unit", "time_period", "date", value="obs_value")
    .drop_nulls("value")
    .sort("geo", "unit", "date")
)
gdp.head()

## 4. Rebuild the viral chart: real GDP indexed to the base quarter

$$\text{index}_t = 100 \times \frac{\text{real GDP}_t}{\text{real GDP}_{\text{base}}}$$

Every country starts at exactly 100, so the chart shows **cumulative growth**, not size.

In [ ]:
real = gdp.filter(pl.col("unit") == REAL_UNIT)
base_vals = real.filter(pl.col("time_period") == BASE).select("geo", base="value")

real = (
    real.join(base_vals, on="geo")
    .sort("geo", "date")
    .with_columns(
        index=100 * pl.col("value") / pl.col("base"),
        qoq=100 * (pl.col("value") / pl.col("value").shift(1) - 1).over("geo"),
        yoy=100 * (pl.col("value") / pl.col("value").shift(4) - 1).over("geo"),
    )
)

(real.sort("date").group_by("country").agg(pl.col("time_period").last(), pl.col("index").last().round(1))
     .sort("index", descending=True))

In [ ]:
def _spread(ys, gap):
    """Nudge end-of-line labels apart so they don't overlap."""
    order = sorted(range(len(ys)), key=lambda i: ys[i])
    out = list(ys)
    for a, b in zip(order, order[1:]):
        if out[b] - out[a] < gap:
            out[b] = out[a] + gap
    return out

def line_chart(df, y, title, ylabel, highlight=HIGHLIGHT, label_fmt="{:.1f}", hline=None, names=None):
    """One line per country; the highlighted country is bold; labels at the right edge."""
    names = names or list(COUNTRIES.values())
    fig, ax = plt.subplots(figsize=(10, 6.5))
    colors = plt.get_cmap("tab10").colors
    ends = []
    for i, name in enumerate(names):
        sub = df.filter(pl.col("country") == name).drop_nulls(y).sort("date")
        if sub.is_empty():
            continue
        bold = name == highlight
        color = "#e8b100" if bold else colors[i % 10]
        ax.plot(sub["date"].to_list(), sub[y].to_list(), color=color,
                lw=3.2 if bold else 1.5, alpha=1 if bold else 0.8, zorder=3 if bold else 2)
        ends.append((name, sub["date"][-1], sub[y][-1], color, bold))
    if hline is not None:
        ax.axhline(hline, color="gray", lw=0.8, ls="--")
    lo, hi = ax.get_ylim()
    label_y = _spread([e[2] for e in ends], gap=(hi - lo) * 0.035)
    for (name, d, v, color, bold), ly in zip(ends, label_y):
        ax.text(d, ly, f"  {name} {label_fmt.format(v)}", ha="left", va="center", color=color,
                fontweight="bold" if bold else "normal", clip_on=False)
    ax.set_title(title, loc="left", fontsize=13)
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.margins(x=0.2)
    plt.tight_layout()
    return ax

since_base = real.filter(pl.col("date") >= real.filter(pl.col("time_period") == BASE)["date"].min())
line_chart(since_base, "index", f"Real GDP, {BASE} = 100",
           f"Index ({REAL_UNIT}, seasonally adjusted)", hline=100);

### Levels: how big is each economy?

Quarterly nominal GDP multiplied by 4 gives an **annualized** rate, shown in trillions of euros.

In [ ]:
nominal = gdp.filter(pl.col("unit") == NOMINAL_UNIT).with_columns(trillions=pl.col("value") * 4 / 1e6)
line_chart(nominal, "trillions", "Nominal GDP, annualized",
           "Trillion euro (current prices)", label_fmt="€{:.2f}T");

In [ ]:
# Snapshot: real GDP at the base and the latest quarter, and how much each country added
last = real["time_period"].max()
(
    real.filter(pl.col("time_period").is_in([BASE, last]))
    .pivot(on="time_period", index="country", values="value")
    .with_columns(
        (pl.col(BASE) * 4 / 1e6).round(2).alias(f"real {BASE} (€T/yr)"),
        (pl.col(last) * 4 / 1e6).round(2).alias(f"real {last} (€T/yr)"),
        ((pl.col(last) - pl.col(BASE)) * 4 / 1e3).round(0).alias("added (€bn/yr)"),
    )
    .drop(BASE, last)
    .sort("added (€bn/yr)", descending=True)
)

### Growth over time

Year-on-year growth (each quarter vs the same quarter a year earlier) shows the COVID collapse, the rebound, and the recent pace. With 10 countries this chart gets busy; try passing `names=["Germany", "Spain", "Poland"]`.

In [ ]:
line_chart(real.filter(pl.col("date") >= pl.date(2017, 1, 1)), "yoy",
           "Real GDP growth, year on year", "% change vs same quarter last year",
           label_fmt="{:+.1f}%", hline=0);

In [ ]:
# Compound average annual growth since the base quarter
(
    real.filter(pl.col("time_period") >= BASE)
    .group_by("country")
    .agg(first=pl.col("value").sort_by("date").first(),
         last=pl.col("value").sort_by("date").last(),
         quarters=pl.len() - 1)
    .with_columns(avg_annual_growth_pct=(100 * ((pl.col("last") / pl.col("first")) ** (4 / pl.col("quarters")) - 1)).round(2))
    .select("country", "quarters", "avg_annual_growth_pct")
    .sort("avg_annual_growth_pct", descending=True)
)

## 5. Does the story survive a per-person view?

Part of Spain's growth since 2019 comes from a growing population (largely immigration), while Germany's and Italy's populations have been flatter. Real GDP **per capita** asks whether the average resident is better off. Eurostat publishes this annually in `nama_10_pc`.

In [ ]:
try:
    pc_raw = fetch_eurostat("nama_10_pc", f"A..B1GQ.{geo_key}", start=START[:4])
    PC_UNIT = newest_unit(pc_raw["unit"].unique().to_list(), "CLV", "_EUR_HAB")
    base_year = BASE[:4]
    pc = (
        pc_raw.filter(pl.col("unit") == PC_UNIT)
        .with_columns(country=name_of(), date=pl.date(pl.col("time_period").cast(pl.Int32), 1, 1))
        .select("country", "geo", "time_period", "date", value="obs_value")
        .drop_nulls("value")
    )
    pc = (pc.join(pc.filter(pl.col("time_period") == base_year).select("geo", base="value"), on="geo")
            .with_columns(index=100 * pl.col("value") / pl.col("base")))
    line_chart(pc.filter(pl.col("time_period") >= base_year), "index",
               f"Real GDP per capita, {base_year} = 100", f"Index ({PC_UNIT})", hline=100)
except Exception as e:
    print("Per-capita section skipped:", e)

## 6. Testing a claim: do Spain, Türkiye and Poland together rival Germany?

A claim going around (this one from a chatbot) says Spain, Türkiye and Poland combined are about 85% of Germany's GDP and grew roughly 3.4% a year over 2015 to 2024 versus about 1% for Germany. Let's check it with Eurostat's annual data.

Two things to watch:

* **Combining countries.** The right way to get a bloc's growth rate is to add up real GDP and compute growth on the total, which automatically weights big members more. Averaging the three growth rates would overweight Poland. (Strictly, chain-linked volumes aren't perfectly additive across countries, but for a comparison like this the error is small.)
* **Currency.** Nominal size in euros (or dollars) depends on exchange rates. Türkiye's lira has lost most of its value over this period, so its euro-denominated size jumps around far more than its real output does.

In [ ]:
real_a = annual.filter(pl.col("unit") == A_REAL).select("geo", "year", real="obs_value")
nom_a = annual.filter(pl.col("unit") == "CP_MEUR").select("geo", "year", nominal="obs_value")
panel = real_a.join(nom_a, on=["geo", "year"], how="left")

bloc_name = " + ".join(NAMES.get(g, g) for g in BLOC)
groups = {NAMES[BENCH]: [BENCH], bloc_name: BLOC}

def group_total(members, label):
    """Sum members' GDP, keeping only years where every member reports."""
    return (panel.filter(pl.col("geo").is_in(members))
            .group_by("year").agg(pl.col("real").sum(), pl.col("nominal").sum(), n=pl.len())
            .filter(pl.col("n") == len(members)).drop("n")
            .with_columns(group=pl.lit(label)).sort("year"))

blocs = pl.concat([group_total(m, k) for k, m in groups.items()])
blocs = blocs.with_columns(growth_pct=100 * (pl.col("real") / pl.col("real").shift(1) - 1).over("group"))

# Each bloc member individually, for reference
members = (panel.filter(pl.col("geo").is_in(BLOC)).sort("geo", "year")
           .with_columns(group=name_of(),
                         growth_pct=100 * (pl.col("real") / pl.col("real").shift(1) - 1).over("geo")))

In [ ]:
G_START, G_END = 2015, 2024   # growth window from the claim

def summarize(df):
    yrs = df.filter(pl.col("year").is_between(G_START, G_END))
    return (yrs.group_by("group").agg(
                avg_growth_pct=pl.col("growth_pct").mean().round(2),
                **{f"growth_{G_END}_pct": pl.col("growth_pct").filter(pl.col("year") == G_END).first().round(2)},
            ))

latest_common = blocs.group_by("group").agg(pl.col("year").max())["year"].min()
size = (blocs.filter(pl.col("year") == latest_common)
        .select("group", nominal_eur_tn=(pl.col("nominal") / 1e6).round(2)))
bench_size = size.filter(pl.col("group") == NAMES[BENCH])["nominal_eur_tn"][0]

table = (size.join(summarize(blocs), on="group")
         .with_columns(pct_of_germany=(100 * pl.col("nominal_eur_tn") / bench_size).round(0)))
print(f"Nominal size in {latest_common}; growth averaged over {G_START}-{G_END}")
pl.concat([table, summarize(members).with_columns(nominal_eur_tn=None, pct_of_germany=None)
                  .select(table.columns)], how="vertical_relaxed")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
base_y = G_START - 1
for label, df, style in [(NAMES[BENCH], blocs.filter(pl.col("group") == NAMES[BENCH]), dict(color="#e8b100", lw=3.2)),
                         (bloc_name, blocs.filter(pl.col("group") == bloc_name), dict(color="black", lw=2.5))]:
    b = df.filter(pl.col("year") == base_y)["real"][0]
    ax.plot(df["year"], 100 * df["real"] / b, label=label, **style)
for g in BLOC:
    df = members.filter(pl.col("geo") == g)
    b = df.filter(pl.col("year") == base_y)["real"]
    if len(b):
        ax.plot(df["year"], 100 * df["real"] / b[0], ls="--", lw=1.2, label=NAMES.get(g, g))
ax.axhline(100, color="gray", lw=0.8, ls=":")
ax.set_title(f"Real GDP, {base_y} = 100", loc="left", fontsize=13)
ax.set_ylabel(f"Index ({A_REAL})")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

## 7. Optional: add the United Kingdom

Eurostat stopped covering the UK after Brexit, but the index chart doesn't need a common currency, so we can pull UK real GDP (chained volume measures, series `ABMI`) from the Office for National Statistics and add it to `since_base`. The level charts would need a currency conversion, so we leave them alone.

In [ ]:
try:
    url = "https://www.ons.gov.uk/generator?format=csv&uri=/economy/grossdomesticproductgdp/timeseries/abmi/qna"
    print("GET", url)
    with urllib.request.urlopen(url, timeout=60) as resp:
        text = resp.read().decode("utf-8")
    rows = [line.replace('"', "").split(",") for line in text.splitlines()]
    uk = pl.DataFrame([r[:2] for r in rows if len(r) >= 2 and " Q" in r[0] and r[0][:4].isdigit()],
                      schema=["period", "value"], orient="row")
    uk = (uk.with_columns(value=pl.col("value").cast(pl.Float64),
                          time_period=pl.col("period").str.replace(" ", "-"))
            .with_columns(date=quarter_to_date, country=pl.lit("United Kingdom"))
            .filter(pl.col("date") >= since_base["date"].min()))
    uk_base = uk.filter(pl.col("time_period") == BASE)["value"][0]
    uk = uk.with_columns(index=100 * pl.col("value") / uk_base)
    with_uk = pl.concat([since_base.select("country", "date", "index"),
                         uk.select("country", "date", "index")])
    line_chart(with_uk, "index", f"Real GDP, {BASE} = 100 (UK from ONS)", "Index",
               hline=100, names=list(COUNTRIES.values()) + ["United Kingdom"])
except Exception as e:
    print("UK section skipped:", e)

## Questions to explore

1. Change `BASE` to `"2015-Q1"` or `"2022-Q4"`. How much of the "Spain beats Germany" story depends on the starting point?
2. Spain's 2020 drop was much deeper than Germany's. How much of its later growth is recovery rather than new growth?
3. In euros, which country *added* the most real output since the base? Does that change the headline?
4. The claim said the trio was "14× faster" than Germany in 2024. What happens to a ratio of growth rates when the denominator is near zero or negative? What would be a better way to express the gap?
5. Compare Türkiye's share of Germany in nominal euros across years. How much of the change is real growth and how much is the exchange rate?
6. Compare the per-capita index with the total index. Which conclusions from the original chart hold up?